# Projeto Final

A odometria é uma técnica amplamente utilizada para estimar, de forma aproximada, a posição e a orientação de um robô móvel ao longo do tempo, com base na medição do deslocamento de suas rodas.

Um dos sensores mais comuns usados para esse fim é o encoder incremental, um dispositivo que conta os giros (ou incrementos de rotação) de cada roda, permitindo calcular a distância percorrida.

A ideia por trás do processo de odometria envolve a integração numérica do movimento do robô ao longo do tempo. É evidente que qualquer erro proveniente dos sensores — como imprecisões na leitura da direção ou distância percorrida — interfere diretamente na determinação da posição. Isso ocorre porque esses erros tendem a se acumular ao longo do tempo, à medida que o robô percorre o caminho em busca de seu destino.

Esse método é bastante viável quando o destino está relativamente próximo do ponto de partida, pois é simples de implementar e fornece uma boa aproximação da posição.

A odometria é normalmente realizada com base nos dados dos encoders localizados nas rodas do robô, assumindo que as revoluções das rodas correspondem a deslocamentos lineares em relação ao solo.

Considere um robô diferencial com duas rodas, cada uma equipada com um encoder incremental. A distância entre as rodas (eixo) $d$ é de $0,4 m$, e o raio de cada roda (direita $r_R$ e esquerda $r_L$) é $0,1 m$, ou seja, $r = r_R = r_L$. 

Assuma que o robô parte da posição inicial ($x_0$, $y_0$) = ($0$, $0$) com orientação $\psi_{0} = 0\ rad$.

<img src="image-1.png" alt="image" size="100%" position="center">

Para robôs móveis com tração diferencial, como o robô uniciclo da figura, as velocidades das rodas podem ser usadas para calcular as velocidades linear e angular do robô em relação ao solo. Que são sinais de controle de alto nível para o robô. Dadas a velocidade angular da roda direita $\omega_R$ e da roda esquerda $\omega_L$, expressa em $rad/s$,  as equações para determinar a velocidade do robô são:

Velocidade linear: $u = r \frac{\omega_R + \omega_L}{2}$
Velocidade angular: $\omega = r \frac{\omega_R - \omega_L}{d}$

Com essas equações, é possível estimar o movimento do robô a partir dos dados de velocidade medidos por encoders instalados nas rodas.

O modelo cinemático de um robô do tipo uniciclo com deslocamento do ponto de controle $h$ é dado por:

$$
\left[
    \begin{array}{c}
    \dot{x} \\
    \dot{y} \\
    \dot{\psi}
    \end{array}
\right]=
\left[
    \begin{array}{cc}
    \cos\psi & -a \sin\psi \\
    \sin\psi & a\cos\psi \\
    0 & 1
    \end{array}
\right]
\left[
    \begin{array}{c}
    u \\
    \omega
    \end{array}
\right]
$$

Onde:

- $(x, y)$: representa a posição do robô no plano cartesiano.
- $\psi$: é o ângulo de orientação do robô em relação ao eixo $x$.
- $u$: é a velocidade linear e $\omega$ é a velocidade angular aplicada ao robô.
- $a$: é a distância entre o eixo virtual que une as rodas do robô e o ponto de aplicação do controle.

Em função dos dados de velocidade das rodas direta e esquerda do robô, determine a rota navegada, sabendo que $a = 0,10$ m.


Para tal, realize os seguintes passos:

1. Realize a leitura do arquivo de dados `RwL.txt`, o qual possui 400 linhas e 3 colunas. Em outras palavras, o arquivo contém 400 medidas de tempo, velocidade da roda direita $\omega_{R}$ e velocidade da roda esquerda $\omega_{L}$.

2. Para demonstrar a evolução temporal das velocidades das rodas, plote os gráficos que relaciona o $tempo \times \omega_{R}$ e o $tempo \times \omega_{L}$.

3. Transforme as velocidades $\omega_{R}$ e $\omega_{L}$ em sinais de controle $u$ e $\omega$

4. Para demonstrar a evolução temporal dos sinais de controle do robô, plote os gráficos que relaciona o $tempo \times u$ e o $tempo \times \omega$.

5. Use o modelo cinemático, para determinar a posição e orientação do robô ao longo do tempo. A título de exemplo, para determinar a orientação do robô, faça $\psi(k) = \frac{\psi(k) - \psi(k-1)}{\Delta t} = \omega$, ou ainda $\psi(k) = \psi(k-1) + \omega \Delta t$.

6. Para demonstrar a evolução temporal da posição e orientação do robô, plote os gráficos que relaciona o $tempo \times x$ e o $tempo \times y$, o &tempo \times \psi$

7. Para demonstrar a navegação do robô no plano cartesiano, plote o gráfico que relaciona $x$ e $y$.

8. Informe numericamente a posição final do robô ao final do experimento.


## 1

Realize a leitura do arquivo de dados `RwL.txt`, o qual possui 400 linhas e 3 colunas. Em outras palavras, o arquivo contém 400 medidas de tempo, velocidade da roda direita $\omega_{R}$ e velocidade da roda esquerda $\omega_{L}$.

In [12]:
import pandas as pd

df = pd.read_csv("wRwL.txt", sep=r"\s+", header=None, names=["t", "omega_R", "omega_L"], engine="python")
print(df.head())

print(df.describe())

df.info()

        t  omega_R  omega_L
0  0.0130   8.3775  -2.2848
1  0.1301   7.0683   0.0915
2  0.2106   5.9269   1.5972
3  0.3106   5.1135   2.4625
4  0.4145   4.5689   2.9364
                t     omega_R     omega_L
count  400.000000  400.000000  400.000000
mean    19.961254    0.837067    1.237123
std     11.561648    1.747158    1.501812
min      0.013000   -8.630300   -2.284800
25%      9.985475    0.106475    0.133600
50%     19.963150    0.557900    0.642750
75%     29.935900    1.707350    1.949300
max     39.912100    8.377500    8.295000
<class 'pandas.DataFrame'>
RangeIndex: 400 entries, 0 to 399
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   t        400 non-null    float64
 1   omega_R  400 non-null    float64
 2   omega_L  400 non-null    float64
dtypes: float64(3)
memory usage: 9.5 KB


## 2

Para demonstrar a evolução temporal das velocidades das rodas, plote os gráficos que relaciona o $tempo \times \omega_{R}$ e o $tempo \times \omega_{L}$.